# Octree Demo

This notebook focuses on octree, lookup, and interpolation behavior.

Scope:

1. Read a data file as a plain `Dataset`
2. Compute per-cell `delta_phi` and inferred refinement levels
3. Build and inspect an octree
4. Run cell lookup in both `xyz` and `(r,polar,azimuth)`
5. Use the interpolator and compare lightly with SciPy


In [ ]:
from pathlib import Path
import sys
import tarfile


HERE = Path.cwd().resolve()
REPO_ROOT = HERE if (HERE / "example_data").is_dir() and (HERE / "batcamp").is_dir() else HERE.parent
if not (REPO_ROOT / "example_data").is_dir() or not (REPO_ROOT / "batcamp").is_dir():
    raise FileNotFoundError("Could not locate repository root with example_data/ and batcamp/.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pooch
import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import LinearNDInterpolator
from batread.dataset import Dataset
from batcamp import Octree
from batcamp import OctreeBuilder
from batcamp import OctreeInterpolator
from batcamp import format_histogram


_G2211_URL = "https://zenodo.org/records/7110555/files/run-Sun-G2211.tar.gz"
_G2211_SHA256 = "c31a32aab08cc20d5b643bba734fd7220e6b369e691f55f88a3a08cc5b2a2136"
_ALLOWED_FILES = {'3d__var_1_n00000000.plt', '3d__var_4_n00044000.plt', '3d__var_4_n00005000.plt'}


def _unique_match(paths: list[Path], *, name: str) -> Path:
    if not paths:
        raise FileNotFoundError(name)
    if len(paths) > 1:
        raise FileNotFoundError(f"Expected unique match for {name}, found {len(paths)} entries: {paths}")
    return paths[0]


def _find_in_example_data(name: str) -> Path:
    return _unique_match(sorted((REPO_ROOT / "example_data").rglob(name)), name=name)


def _fetch_from_g2211_archive(name: str) -> Path:
    archive_path = Path(
        pooch.retrieve(
            url=_G2211_URL,
            known_hash=_G2211_SHA256,
            progressbar=False,
        )
    )
    with tarfile.open(archive_path, "r:gz") as tar:
        member_names = sorted(
            m.name for m in tar.getmembers() if m.isfile() and Path(m.name).name == name
        )
    member = _unique_match([Path(m) for m in member_names], name=name).as_posix()
    extracted = pooch.retrieve(
        url=_G2211_URL,
        known_hash=_G2211_SHA256,
        progressbar=False,
        processor=pooch.Untar(members=[member]),
    )
    if isinstance(extracted, (list, tuple)):
        extracted = extracted[0]
    return Path(extracted)


def data_file(name: str) -> Path:
    """Resolve one allowed sample file from local example_data or pooch."""
    if name not in _ALLOWED_FILES:
        raise ValueError(f"Unsupported sample file '{name}'. Allowed: {sorted(_ALLOWED_FILES)}")
    try:
        return _find_in_example_data(name)
    except FileNotFoundError:
        return _fetch_from_g2211_archive(name)





### Pick the mixed-level 3D file used during development.


In [ ]:
path = data_file('3d__var_4_n00044000.plt')
path


### Plain Dataset read (no SmartDs)


In [ ]:
ds = Dataset.from_file(str(path))
print(ds)


### Build octree and inspect inferred levels


In [ ]:
tree = Octree.from_dataset(ds, tree_coord='rpa')
builder = OctreeBuilder()
delta_phi, center_phi, cell_levels, expected_delta_phi, coarse_delta_phi = builder.compute_phi_levels(ds)
print('coarsest delta_phi [rad]:', coarse_delta_phi)
print('cell level histogram:', format_histogram(cell_levels))


### Octree summary


In [ ]:
print(tree)


### Lookup in xyz and rpa


In [ ]:
lookup = tree.lookup
q_xyz = (1.0, 0.0, 0.0)
hit_xyz = lookup.lookup_point(np.array(q_xyz, dtype=float), space="xyz")
print('lookup xyz:', q_xyz, '->', hit_xyz)
r, polar, azimuth = tree.xyz_to_rpa(np.asarray(q_xyz, dtype=float))
hit_rpa = lookup.lookup_point(np.array([r, polar, azimuth], dtype=float), space="rpa")
print('lookup rpa:', (r, polar, azimuth), '->', hit_rpa)
print('same cell id?', hit_xyz is not None and hit_rpa is not None and hit_xyz.cell_id == hit_rpa.cell_id)


### Interpolator usage


In [ ]:
interp = OctreeInterpolator(
    ds,
    ['Rho [g/cm^3]'],
    tree=tree,
)
queries = np.array([
    [1.0, 0.0, 0.0],
    [5.0, 0.0, 0.0],
    [10.0, 1.0, 0.5],
], dtype=float)
vals, cell_ids = interp(queries, return_cell_ids=True)
print('queries:')
print(queries)
print('values:')
print(vals)
print('cell ids:')
print(cell_ids)


### Optional: lightweight comparison to SciPy LinearNDInterpolator.

To keep notebook runtime reasonable, this uses a point subset.


In [ ]:
xyz = np.column_stack([
    np.asarray(ds.variable('X [R]'), dtype=float),
    np.asarray(ds.variable('Y [R]'), dtype=float),
    np.asarray(ds.variable('Z [R]'), dtype=float),
])
vv = np.asarray(ds.variable('Rho [g/cm^3]'), dtype=float)
rng = np.random.default_rng(0)
n_sub = min(40000, xyz.shape[0])
sub = rng.choice(xyz.shape[0], size=n_sub, replace=False)
lin = LinearNDInterpolator(xyz[sub], vv[sub], fill_value=np.nan)
q = xyz[rng.choice(xyz.shape[0], size=2000, replace=False)]
ours = interp(q)
scipy_vals = lin(q)
mask = np.isfinite(ours) & np.isfinite(scipy_vals)
print('scipy subset size:', n_sub)
print('finite overlap fraction:', float(np.mean(mask)))
if np.any(mask):
    ad = np.abs(ours[mask] - scipy_vals[mask])
    print('max abs diff:', float(np.max(ad)))
    print('median abs diff:', float(np.median(ad)))
